Preprocessing

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# Config
# ==========================================
raw_folder = 'data/raw_light_curves'
save_folder = 'data/preprocessed_light_curves'
catalog_file = 'data/DB_QSO_S82.dat'
os.makedirs(save_folder, exist_ok=True)

# ==========================================
# Load catalog and compute A_r from Au
# ==========================================
cat = pd.read_csv(catalog_file, sep=r'\s+', engine='python')
cat['SDR5ID'] = cat['SDR5ID'].astype(str)
cat['A_r'] = 0.534 * cat['Au']
ext_dict = dict(zip(cat['SDR5ID'], cat['A_r']))

# ==========================================
# Clean and correct
# ==========================================
def clean_and_correct(file_path, A_r):
    df = pd.read_csv(file_path, sep=None, engine='python')
    mag_col = [c for c in df.columns if 'mag' in c.lower()][0]

    # Drop -99.99 sentinel values
    df = df[df[mag_col] != -99.99].copy()
    df = df[df[mag_col] != 99.99].copy()
    df.reset_index(drop=True, inplace=True)

    if len(df) == 0:
        return df

    # Extinction correction: de-redden
    if A_r is not None:
        df[mag_col] = df[mag_col] - A_r

    return df

# ==========================================
# Execution
# ==========================================
all_files = glob.glob(os.path.join(raw_folder, "*"))
print(f"Processing {len(all_files)} files...")

missing = 0
for file_path in all_files:
    if os.path.isdir(file_path):
        continue
    qso_id = os.path.splitext(os.path.basename(file_path))[0]

    A_r = ext_dict.get(qso_id, None)
    if A_r is None:
        missing += 1

    try:
        df_clean = clean_and_correct(file_path, A_r)
        if len(df_clean) > 0:
            out_path = os.path.join(save_folder, f"{qso_id}.csv")
            df_clean.to_csv(out_path, index=False)
    except Exception as e:
        print(f"Error processing {qso_id}: {e}")

print(f"Done. {missing} files had no matching extinction value.")
print(f"Output: {save_folder}")

Removing Single Point Spikes (MAD)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# ==========================================
# 1. Config
# ==========================================
raw_folder = r'data/preprocessed_light_curves'
save_folder = r'data/cleaned_light_curves'
os.makedirs(save_folder, exist_ok=True)

SIGMA_THRESH = 5.0  

# ==========================================
# 2. Vectorized Interpolation Cleaning
# ==========================================
def clean_vetted_data(file_path):
    # Load and identify columns
    df = pd.read_csv(file_path, sep=None, engine='python')
    time_col = 'MJD_r' if 'MJD_r' in df.columns else 'MJD'
    mag_col = 'mag_r' if 'mag_r' in df.columns else 'mag'
    
    # Sort by time
    df = df.sort_values(time_col).reset_index(drop=True)
    mags = df[mag_col].values
    n = len(mags)

    if n < 3:
        return df.copy(), 0

    # Robust global scatter estimate for scale
    median_mag = np.median(mags)
    mad = np.median(np.abs(mags - median_mag))
    sigma_robust = mad / 0.6745 + 1e-6

    # --- Vectorized Continuity Check ---
    # Calculate 'expected' as average of i-1 and i+1 for all interior points
    # mags[0:-2] is i-1, mags[2:] is i+1
    expected = 0.5 * (mags[0:-2] + mags[2:])
    
    # Calculate deviation for interior points mags[1:-1]
    actual = mags[1:-1]
    deviation = np.abs(actual - expected)
    
    # Create mask for interior points
    interior_keep = deviation <= (SIGMA_THRESH * sigma_robust)
    
    # Construct final keep mask (always keep first and last point for safety)
    keep_mask = np.concatenate(([True], interior_keep, [True]))

    clean_df = df[keep_mask].copy()
    num_removed = n - len(clean_df)

    return clean_df, num_removed

# ==========================================
# 3. Execution
# ==========================================
all_files = glob.glob(os.path.join(raw_folder, "*"))

print(f"Starting interpolation-based cleaning on {len(all_files)} files...")

for file_path in all_files:
    if os.path.isdir(file_path): continue
    qso_id = os.path.basename(file_path)

    try:
        df_clean, removed = clean_vetted_data(file_path)
        out_path = os.path.join(save_folder, f"{qso_id}.csv")
        df_clean.to_csv(out_path, index=False)
    except Exception as e:
        print(f"Error processing {qso_id}: {e}")

print(f"Cleaning complete. Output saved to: {save_folder}")